In [ ]:
# ============================================================
#  REVİZE EDİLMİŞ KOD — Q1 Hakem Önerilerine Göre
#  Önerilen Model: H/H0 = a + b·(S/S0) + c·(S/S0)² + d·√(S/S0)
#  Yeni Eklemeler:
#  1. MAPE + MBE metrikleri
#  2. Walk-forward (expanding window) cross-validation
#  3. Bootstrap güven aralıkları (katsayılar için)
#  4. Cohen's d etki büyüklüğü (PSO vs POA)
#  5. TFT için çoklu-seed kararlılık analizi
#  6. Geliştirilmiş yakınsama + residual grafikler
# ============================================================

import os, sys, glob, json, math, textwrap, subprocess, warnings, random, re
warnings.filterwarnings("ignore")

# ── 0. PAKETLER ──────────────────────────────────────────────
def ensure(pkg, pipname=None):
    pipname = pipname or pkg
    try:
        __import__(pkg)
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", pipname])

for p, n in [("openpyxl","openpyxl"),("pandas","pandas"),("numpy","numpy"),
             ("matplotlib","matplotlib"),("seaborn","seaborn"),
             ("scipy","scipy"),("sklearn","scikit-learn")]:
    ensure(p, n)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import least_squares
from scipy.stats import ttest_rel, wilcoxon, levene
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 130
np.set_printoptions(suppress=True, precision=6)
random.seed(42); np.random.seed(42)

# ── 1. DARTS / TFT ───────────────────────────────────────────
RUN_TFT = True
HAS_DARTS = False
TFT_IMPORT_ERROR = None

if RUN_TFT:
    try:
        from darts import TimeSeries
        from darts.models import TFTModel
        from darts.dataprocessing.transformers import Scaler
        from darts.utils.timeseries_generation import datetime_attribute_timeseries
        HAS_DARTS = True
    except Exception:
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "-q",
                                   "install", "u8darts[torch]>=0.30.0"])
            from darts import TimeSeries
            from darts.models import TFTModel
            from darts.dataprocessing.transformers import Scaler
            from darts.utils.timeseries_generation import datetime_attribute_timeseries
            HAS_DARTS = True
        except Exception as e:
            HAS_DARTS = False
            TFT_IMPORT_ERROR = repr(e)

# ── 2. AYARLAR ────────────────────────────────────────────────
TEST_SIZE    = 12
N_SEEDS      = 20
N_SEEDS_TFT  = 20
SEED0        = 16
N_BOOTSTRAP  = 500
WF_MIN_TRAIN = 36

PSO_CFG = dict(pop=40, iters=500, w_start=0.9, w_end=0.4, c1=1.5, c2=2.0)
POA_CFG = dict(pop=40, iters=500)

TFT_CFG = dict(
    input_chunk_length=24,
    output_chunk_length=1,
    hidden_size=4,
    lstm_layers=1,
    num_attention_heads=1,
    dropout=0.05,
    batch_size=4,
    n_epochs=300,
)

OUTPUT_ROOT       = "/content/review_evidence_pack"
DRIVE_OUTPUT_ROOT = "/content/drive/MyDrive/review_evidence_pack"
SAVE_TO_DRIVE_TOO = True

# ── 4. YARDIMCI FONKSİYONLAR ─────────────────────────────────
def safe_mkdir(path):
    os.makedirs(path, exist_ok=True)
    return path

def save_csv(df, path):
    df.to_csv(path, index=False, encoding="utf-8-sig")

def save_df_as_png(df, path, title=None, fontsize=9, max_rows=40):
    dfx = df.copy().head(max_rows)
    fig_h = max(2.2, 0.45*(len(dfx)+2))
    fig_w = max(9, 1.25*len(dfx.columns))
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    ax.axis("off")
    if title:
        ax.set_title(title, fontsize=12, fontweight="bold", pad=12)
    tbl = ax.table(cellText=dfx.values, colLabels=dfx.columns,
                   loc="center", cellLoc="center")
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(fontsize)
    tbl.scale(1, 1.25)
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight")
    plt.close()

def positive_clip(x, eps=1e-8):
    return np.clip(np.asarray(x, float), eps, None)

# ── 3. DRIVE MOUNT ───────────────────────────────────────────
try:
    from google.colab import drive, files
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    try:
        drive.mount("/content/drive", force_remount=False)
    except Exception:
        pass

os.makedirs(OUTPUT_ROOT, exist_ok=True)

subdirs = [
    "tables",
    "figs",
    "residuals",
    "convergence",
    "tft",
]

for sd in subdirs:
    safe_mkdir(os.path.join(OUTPUT_ROOT, sd))
    if SAVE_TO_DRIVE_TOO:
        safe_mkdir(os.path.join(DRIVE_OUTPUT_ROOT, sd))



# ── 5. METRİKLER ─────────────────────────────────────────────
def metrics(y_true, y_pred):
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)
    mask = y_true != 0
    mape = float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100)
    mbe  = float(np.mean(y_pred - y_true))
    return {
        "R2"  : float(r2_score(y_true, y_pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "MAE" : float(mean_absolute_error(y_true, y_pred)),
        "MAPE": mape,
        "MBE" : mbe,
    }

# ── 6. VERİ YÜKLEME ──────────────────────────────────────────
COMMON_EXCEL_PATTERNS = [
    "/content/drive/MyDrive/solar_gunler_Serkan.xlsx",
    "/content/drive/MyDrive/solar_aylar_Serkan.xlsx",
    "/content/drive/MyDrive/solaraylarSerkan.xlsx",
    "/content/drive/MyDrive/solargunlerSerkan-2.xlsx",
    "/content/*.xlsx",
    "/content/drive/MyDrive/*.xlsx",
]

def resolve_excel_path():
    for pat in COMMON_EXCEL_PATTERNS:
        matches = glob.glob(pat, recursive=True)
        if matches:
            matches = sorted(matches, key=lambda x: len(x))
            print(f"✓ Excel bulundu: {matches[0]}")
            return matches[0]
    if IN_COLAB:
        print("Excel bulunamadı, lütfen yükleyin...")
        up = files.upload()
        if not up:
            raise FileNotFoundError("Excel yüklenmedi.")
        return f"/content/{list(up.keys())[0]}"
    raise FileNotFoundError("Excel bulunamadı.")

EXCEL_PATH = resolve_excel_path()

raw_df = pd.read_excel(EXCEL_PATH)
raw_df.columns = [str(c).strip() for c in raw_df.columns]

cols       = list(raw_df.columns)
year_col   = cols[0]
month_col  = cols[1]
day_col    = cols[2]
H_col      = cols[3]
sun_col    = cols[4]
daylen_col = cols[5]
H0_col     = cols[6]
SSO_col    = cols[7]
HHO_col    = cols[8]

print("✓ Sütun eşlemesi:")
for name, col in zip(["year","month","day","H","S","S0","H0","S/S0","H/H0"],
                     [year_col,month_col,day_col,H_col,sun_col,
                      daylen_col,H0_col,SSO_col,HHO_col]):
    print(f"   {name:6} → {col}")

df       = raw_df.copy()
num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]

monthly_df = df.groupby([year_col, month_col], as_index=False)[num_cols]\
               .mean(numeric_only=True)

monthly_df = monthly_df.rename(columns={
    year_col:  "year",
    month_col: "month",
    H_col:     "H",
    H0_col:    "H0",
    SSO_col:   "SoverS0",
    HHO_col:   "HoverH0",
})

monthly_df["date"] = pd.to_datetime(dict(
    year  = monthly_df["year"].astype(int),
    month = monthly_df["month"].astype(int),
    day   = 1
))
monthly_df = monthly_df.sort_values("date").reset_index(drop=True)

for col in ["H", "H0", "SoverS0", "HoverH0"]:
    monthly_df[col] = pd.to_numeric(monthly_df[col], errors="coerce")

monthly_df = monthly_df.dropna(
    subset=["date","H","H0","SoverS0","HoverH0"]
).reset_index(drop=True)

train_df = monthly_df.iloc[:-TEST_SIZE].copy()
test_df  = monthly_df.iloc[-TEST_SIZE:].copy()

x_train       = train_df["SoverS0"].values.astype(float)
x_test        = test_df["SoverS0"].values.astype(float)
y_ratio_train = train_df["HoverH0"].values.astype(float)
y_ratio_test  = test_df["HoverH0"].values.astype(float)
H_train       = train_df["H"].values.astype(float)
H_test        = test_df["H"].values.astype(float)
H0_train      = train_df["H0"].values.astype(float)
H0_test       = test_df["H0"].values.astype(float)

print(f"\n✓ Veri hazır: {len(monthly_df)} aylık gözlem  "
      f"[{monthly_df['date'].min().date()} → {monthly_df['date'].max().date()}]")
print(f"   Eğitim: {len(train_df)}  |  Test: {len(test_df)}")

# ── 7. MODEL FONKSİYONLARI ───────────────────────────────────
def f_linear(x, p):
    a, b = p
    return a + b*x

def f_poly2(x, p):
    a, b, c = p
    return a + b*x + c*x**2

def f_poly3(x, p):
    a, b, c, d = p
    return a + b*x + c*x**2 + d*x**3

def f_exp(x, p):
    a, b = p
    return a * np.exp(np.clip(b*x, -50, 50))

def f_log(x, p):
    a, b = p
    return a + b*np.log(positive_clip(x))

def f_power(x, p):
    a, b = p
    return a * positive_clip(x)**b

def f_prop_sqrt(x, p):
    # Önerilen model — Horzum (2024) bildiri formülü
    # H/H0 = a + b·(S/S0) + c·(S/S0)² + d·√(S/S0)
    a, b, c, d = p
    return a + b*x + c*x**2 + d*np.sqrt(positive_clip(x))

def f_weibull(x, p):
    a, b, c, d = p
    return a * positive_clip(x)**b * np.exp(np.clip(-c * positive_clip(x)**d, -50, 50))

# ── 8. MODEL TANIMLARI ────────────────────────────────────────
MODEL_SPECS = {
    "Linear"      : dict(fn=f_linear,    dim=2, bounds=([-5,-5],[5,5]),                   origin="paper"),
    "Polynomial2" : dict(fn=f_poly2,     dim=3, bounds=([-5,-10,-10],[5,10,10]),          origin="paper"),
    "Polynomial3" : dict(fn=f_poly3,     dim=4, bounds=([-5,-10,-10,-10],[5,10,10,10]),   origin="paper"),
    "Exponential" : dict(fn=f_exp,       dim=2, bounds=([-5,-10],[5,10]),                 origin="paper"),
    "Logarithmic" : dict(fn=f_log,       dim=2, bounds=([-5,-10],[5,10]),                 origin="paper"),
    "Power"       : dict(fn=f_power,     dim=2, bounds=([-5,-10],[5,10]),                 origin="paper"),
    "Proposed_sqrt":dict(fn=f_prop_sqrt, dim=4, bounds=([-5,-50,-50,-50],[5,50,50,50]), origin="proposed"),
    "Weibull"     : dict(fn=f_weibull,   dim=4, bounds=([-10,-10,-10,-10],[10,10,10,10]), origin="notebook"),
}

RUN_MODELS  = list(MODEL_SPECS.keys())
RUN_METHODS = ["EKK", "PSO", "POA"]

# ── 9. OPTİMİZATÖRLER ────────────────────────────────────────
def safe_pred(fn, x, p):
    v = fn(x, p)
    v = np.asarray(v, float)
    v[~np.isfinite(v)] = np.nan
    return v

def sse_ratio(p, fn, x, y):
    pred = safe_pred(fn, x, p)
    if np.any(np.isnan(pred)): return 1e20
    return float(np.sum((pred - y)**2))

def fit_ekk(spec, x, y, seed=42):
    lb, ub = map(np.array, spec["bounds"])
    rng = np.random.default_rng(seed)
    x0  = np.clip(rng.uniform(lb, ub), lb + 1e-6, ub - 1e-6)
    res = least_squares(lambda p: safe_pred(spec["fn"], x, p) - y,
                        x0, bounds=(lb, ub), max_nfev=20000)
    return res.x, float(np.sum(res.fun**2)), None

def pso_optimize(fn, bounds, x, y, seed=42, **cfg):
    lb, ub = map(np.array, bounds)
    dim    = len(lb)
    pop, iters = cfg.get("pop", 40), cfg.get("iters", 500)
    w_s, w_e   = cfg.get("w_start", 0.9), cfg.get("w_end", 0.4)
    c1, c2     = cfg.get("c1", 1.5), cfg.get("c2", 2.0)
    rng = np.random.default_rng(seed)
    X   = rng.uniform(lb, ub, (pop, dim))
    V   = np.zeros((pop, dim))
    P   = X.copy()
    P_fit = np.array([sse_ratio(p, fn, x, y) for p in X])
    gi  = np.argmin(P_fit); G = P[gi].copy(); G_fit = float(P_fit[gi])
    history = []
    for t in range(iters):
        w = w_s + (w_e - w_s) * t / max(1, iters - 1)
        for i in range(pop):
            r1, r2 = rng.random(dim), rng.random(dim)
            V[i] = w*V[i] + c1*r1*(P[i]-X[i]) + c2*r2*(G-X[i])
            X[i] = np.clip(X[i] + V[i], lb, ub)
            f = sse_ratio(X[i], fn, x, y)
            if f < P_fit[i]: P_fit[i] = f; P[i] = X[i].copy()
            if f < G_fit:    G_fit = float(f); G = X[i].copy()
        history.append(G_fit)
    return G, G_fit, history

def poa_optimize(fn, bounds, x, y, seed=42, **cfg):
    lb, ub     = map(np.array, bounds)
    dim        = len(lb)
    pop, iters = cfg.get("pop", 40), cfg.get("iters", 500)
    rng = np.random.default_rng(seed)
    X   = rng.uniform(lb, ub, (pop, dim))
    fit = np.array([sse_ratio(xx, fn, x, y) for xx in X])
    bi  = np.argmin(fit); best = X[bi].copy(); best_fit = float(fit[bi])
    history = []
    for t in range(iters):
        for i in range(pop):
            I    = rng.integers(1, 3)
            cand = np.clip(X[i] + rng.random(dim) * (best - I*X[i]), lb, ub)
            f    = sse_ratio(cand, fn, x, y)
            if f < fit[i]: X[i] = cand; fit[i] = f
            if f < best_fit: best = cand.copy(); best_fit = float(f)
        R = 0.2 * (1 - (t + 1) / iters)
        for i in range(pop):
            cand = np.clip(X[i] + (2*rng.random(dim)-1) * R*(ub-lb), lb, ub)
            f    = sse_ratio(cand, fn, x, y)
            if f < fit[i]: X[i] = cand; fit[i] = f
            if f < best_fit: best = cand.copy(); best_fit = float(f)
        history.append(best_fit)
    return best, best_fit, history

def fit_by_method(method, spec, x, y, seed=42):
    if method == "EKK": return fit_ekk(spec, x, y, seed)
    if method == "PSO": return pso_optimize(spec["fn"], spec["bounds"], x, y, seed, **PSO_CFG)
    if method == "POA": return poa_optimize(spec["fn"], spec["bounds"], x, y, seed, **POA_CFG)

# ── 10. TÜM MODELLERİ ÇALIŞTIR ───────────────────────────────
print("\n── EMPİRİK MODEL EĞİTİMİ ──")
all_rows, pred_store, history_store, param_store = [], {}, {}, {}

for model_name in RUN_MODELS:
    spec = MODEL_SPECS[model_name]
    for method in RUN_METHODS:
        try:
            params, fit_val, history = fit_by_method(
                method, spec, x_train, y_ratio_train, SEED0)
            pr_tr   = safe_pred(spec["fn"], x_train, params)
            pr_te   = safe_pred(spec["fn"], x_test,  params)
            H_pr_tr = pr_tr * H0_train
            H_pr_te = pr_te * H0_test
            mt_tr   = metrics(H_train, H_pr_tr)
            mt_te   = metrics(H_test,  H_pr_te)
            all_rows.append({
                "Model": model_name, "Method": method, "Origin": spec["origin"],
                "n_params": spec["dim"],
                **{f"train_{k}": v for k, v in mt_tr.items()},
                **{f"test_{k}":  v for k, v in mt_te.items()},
                "obj_train_SSE_ratio": fit_val,
                "params": np.round(params, 6).tolist()
            })
            pred_store[(model_name, method)]    = {"tr":pr_tr,"te":pr_te,"H_tr":H_pr_tr,"H_te":H_pr_te}
            history_store[(model_name, method)] = history
            param_store[(model_name, method)]   = params
            print(f"  {model_name:<16} {method}")
        except Exception as e:
            print(f"  {model_name:<16} {method}  HATA: {repr(e)[:60]}")
            all_rows.append({
                "Model": model_name, "Method": method, "Origin": spec["origin"],
                "n_params": spec["dim"],
                **{f"train_{k}": np.nan for k in ["R2","RMSE","MAE","MAPE","MBE"]},
                **{f"test_{k}":  np.nan for k in ["R2","RMSE","MAE","MAPE","MBE"]},
                "obj_train_SSE_ratio": np.nan, "params": repr(e)
            })

results_df = (pd.DataFrame(all_rows)
              .sort_values(["test_R2","test_RMSE"], ascending=[False,True])
              .reset_index(drop=True))

# ── 11. WALK-FORWARD CROSS-VALIDATION ────────────────────────
print("\n── WALK-FORWARD CROSS-VALIDATION ──")
wf_models  = ["Proposed_sqrt", "Linear", "Polynomial3"]
wf_methods = ["PSO", "POA", "EKK"]
wf_rows    = []

total_n = len(monthly_df)
for fold_end in range(WF_MIN_TRAIN + 1, total_n - TEST_SIZE):
    wf_train    = monthly_df.iloc[:fold_end]
    wf_test_row = monthly_df.iloc[fold_end]
    xt     = wf_train["SoverS0"].values.astype(float)
    yt     = wf_train["HoverH0"].values.astype(float)
    x1     = np.array([wf_test_row["SoverS0"]])
    H0_1   = wf_test_row["H0"]
    H_true = wf_test_row["H"]
    for mname in wf_models:
        spec = MODEL_SPECS[mname]
        for method in wf_methods:
            try:
                params, _, _ = fit_by_method(method, spec, xt, yt, seed=42)
                H_pred = float(safe_pred(spec["fn"], x1, params)[0] * H0_1)
                wf_rows.append({
                    "fold": fold_end, "model": mname, "method": method,
                    "H_true": H_true, "H_pred": H_pred,
                    "error": H_pred - H_true,
                    "abs_error": abs(H_pred - H_true)
                })
            except:
                pass

wf_df = pd.DataFrame(wf_rows)
wf_summary = (wf_df.groupby(["model","method"])
              .apply(lambda g: pd.Series({
                  "WF_RMSE": np.sqrt(np.mean(g["error"]**2)),
                  "WF_MAE" : np.mean(g["abs_error"]),
                  "WF_MBE" : np.mean(g["error"]),
                  "WF_MAPE": np.mean(np.abs(g["error"]) / np.abs(g["H_true"])) * 100,
                  "n_folds": len(g)
              }), include_groups=False)
              .reset_index())
print(wf_summary.round(2).to_string(index=False))



# ── 12. BOOTSTRAP GÜVENİLİRLİK ARALIĞI ───────────────────────
print("\n── BOOTSTRAP GÜVEN ARALIĞI (Proposed_sqrt + PSO) ──")
# Not: Geniş CI, eşdeğer çözüm kümesinin (equifinality) göstergesidir.
# b ve d katsayıları birlikte model performansını belirler.
best_model_name = "Proposed_sqrt"
best_method     = "PSO"
spec_bs         = MODEL_SPECS[best_model_name]   # güncel bounds kullanılır
COEF_NAMES      = ["a", "b", "c", "d"]           # 4 parametre

bs_params = []
rng_bs = np.random.default_rng(0)
for b in range(N_BOOTSTRAP):
    idx = rng_bs.integers(0, len(x_train), len(x_train))
    try:
        p, _, _ = fit_by_method(best_method, spec_bs,
                                x_train[idx], y_ratio_train[idx], seed=int(b))
        bs_params.append(p)
    except:
        pass

bs_params = np.array(bs_params)
bs_df = pd.DataFrame(bs_params,
                     columns=[f"coef_{c}" for c in COEF_NAMES])
ci_df = pd.DataFrame({
    "coef"   : COEF_NAMES,
    "mean"   : bs_df.mean().values,
    "std"    : bs_df.std().values,
    "ci_low" : np.percentile(bs_params, 2.5,  axis=0),
    "ci_high": np.percentile(bs_params, 97.5, axis=0),
})
print(ci_df.round(4).to_string(index=False))
print("\n[Not] b ve d için geniş CI, equifinality'nin göstergesidir.")
print("      Model test performansı tüm seed'lerde tutarlıdır (bkz. çoklu-seed analizi).")

# ── 13. ÇOKLU-SEED STABİLİTE (PSO + POA) ─────────────────────
print("\n── ÇOKLU-SEED STABİLİTE ANALİZİ ──")
multi_rows, hist_rows = [], []
prop_key = "Proposed_sqrt"

for method in ["PSO", "POA"]:
    spec = MODEL_SPECS[prop_key]
    best_rank, best_pred = None, None
    for seed in range(1, N_SEEDS + 1):
        params, fit_val, history = fit_by_method(
            method, spec, x_train, y_ratio_train, seed)
        pr_te   = safe_pred(spec["fn"], x_test,  params)
        H_pr_te = pr_te  * H0_test
        pr_tr   = safe_pred(spec["fn"], x_train, params)
        H_pr_tr = pr_tr  * H0_train
        mt_tr   = metrics(H_train, H_pr_tr)
        mt_te   = metrics(H_test,  H_pr_te)
        row = {"method": method, "seed": seed,
               **{f"coef_{COEF_NAMES[i]}": params[i] for i in range(len(params))},
               **{f"train_{k}": v for k, v in mt_tr.items()},
               **{f"test_{k}":  v for k, v in mt_te.items()},
               "obj_train_SSE_ratio": fit_val}
        multi_rows.append(row)
        if history:
            for it, val in enumerate(history, 1):
                hist_rows.append({"method":method,"seed":seed,
                                  "iter":it,"best_sse":float(val)})
        rank = (mt_te["R2"], -mt_te["RMSE"])
        if best_rank is None or rank > best_rank:
            best_rank = rank
            best_pred = H_pr_te.copy()
            pred_store[(prop_key, f"{method}_best")] = {"H_te": best_pred}
        print(f"  {method}  seed={seed:2d}  "
              f"R²={mt_te['R2']:.4f}  RMSE={mt_te['RMSE']:.2f}  "
              f"MAPE={mt_te['MAPE']:.2f}%  MBE={mt_te['MBE']:.2f}")

multi_df = pd.DataFrame(multi_rows)
hist_df  = pd.DataFrame(hist_rows)

summary_df = (multi_df.groupby("method", as_index=False)
              .agg(
                  n_runs    =("seed",      "count"),
                  mean_R2   =("test_R2",   "mean"),
                  std_R2    =("test_R2",   "std"),
                  min_R2    =("test_R2",   "min"),
                  max_R2    =("test_R2",   "max"),
                  mean_RMSE =("test_RMSE", "mean"),
                  std_RMSE  =("test_RMSE", "std"),
                  mean_MAE  =("test_MAE",  "mean"),
                  std_MAE   =("test_MAE",  "std"),
                  mean_MAPE =("test_MAPE", "mean"),
                  mean_MBE  =("test_MBE",  "mean"),
              ))

# ── EN İYİ SEED KATSAYILARI ──
best_pso = multi_df[multi_df['method']=='PSO'].sort_values('test_R2', ascending=False).iloc[0]
best_poa = multi_df[multi_df['method']=='POA'].sort_values('test_R2', ascending=False).iloc[0]

print("=== EN İYİ PSO KATSAYILARI ===")
print(f"Seed : {int(best_pso['seed'])}")
print(f"R²   : {best_pso['test_R2']:.4f}")
print(f"RMSE : {best_pso['test_RMSE']:.2f}")
print(f"a={best_pso['coef_a']:.6f}, b={best_pso['coef_b']:.6f}, "
      f"c={best_pso['coef_c']:.6f}, d={best_pso['coef_d']:.6f}")

print("\n=== EN İYİ POA KATSAYILARI ===")
print(f"Seed : {int(best_poa['seed'])}")
print(f"R²   : {best_poa['test_R2']:.4f}")
print(f"RMSE : {best_poa['test_RMSE']:.2f}")
print(f"a={best_poa['coef_a']:.6f}, b={best_poa['coef_b']:.6f}, "
      f"c={best_poa['coef_c']:.6f}, d={best_poa['coef_d']:.6f}")

# ── 14. İSTATİSTİKSEL TEST + COHEN'S D ───────────────────────
def paired_stats(df, metric, higher_better=True):
    pso = df[df["method"]=="PSO"].sort_values("seed")[f"test_{metric}"].values
    poa = df[df["method"]=="POA"].sort_values("seed")[f"test_{metric}"].values
    diff = poa - pso if higher_better else pso - poa
    pooled_std = np.sqrt((np.std(pso,ddof=1)**2 + np.std(poa,ddof=1)**2) / 2)
    cohens_d   = float(np.mean(diff) / pooled_std) if pooled_std > 0 else np.nan
    out = {"metric": metric,
           "PSO_mean": float(np.mean(pso)), "POA_mean": float(np.mean(poa)),
           "PSO_std":  float(np.std(pso, ddof=1)),
           "POA_std":  float(np.std(poa, ddof=1)),
           "delta":    float(np.mean(diff)),
           "cohens_d": cohens_d,
           "ttest_p": np.nan, "wilcoxon_p": np.nan, "levene_p": np.nan}
    try: out["ttest_p"]    = float(ttest_rel(poa,pso).pvalue if higher_better
                                   else ttest_rel(pso,poa).pvalue)
    except: pass
    try: out["wilcoxon_p"] = float(wilcoxon(diff).pvalue)
    except: pass
    try: out["levene_p"]   = float(levene(pso, poa).pvalue)
    except: pass
    return out

stats_df = pd.DataFrame([
    paired_stats(multi_df, "R2",   True),
    paired_stats(multi_df, "RMSE", False),
    paired_stats(multi_df, "MAE",  False),
    paired_stats(multi_df, "MAPE", False),
    paired_stats(multi_df, "MBE",  False),
])
print("\nİstatistiksel Testler (Cohen's d dahil):")
print(stats_df[["metric","PSO_mean","POA_mean","delta","cohens_d",
                "ttest_p","wilcoxon_p"]].round(4).to_string(index=False))

# ── İSTATİSTİKSEL GÜÇ ANALİZİ ──
# Kodu çalıştırdıktan sonra yeni hücreye ekle (stats_df mevcut olmalı)

from statsmodels.stats.power import TTestIndPower

analysis = TTestIndPower()
# Gerçek Cohen's d değerlerini stats_df'den al
for _, row in stats_df.iterrows():
    d = abs(row['cohens_d'])
    power = analysis.solve_power(effect_size=d, nobs1=20,
                                 alpha=0.05, alternative='two-sided')
    print(f"Metrik={row['metric']:6}  Cohen's d={d:.4f}  "
          f"İstatistiksel güç={power:.3f}  (n=20)")

# ── MAPE ÖLÇEK KONTROLÜ ──────────────────────────
mape_H = metrics(H_test, pred_store[('Proposed_sqrt','PSO')]['H_te'])['MAPE']

pred_ratio_pso = pred_store[('Proposed_sqrt','PSO')]['te']
mape_ratio = float(np.mean(
    np.abs((y_ratio_test - pred_ratio_pso) / y_ratio_test)) * 100)

print(f"Proposed_sqrt PSO — H ölçeğinde MAPE    : {mape_H:.3f}%")
print(f"Proposed_sqrt PSO — H/H0 ölçeğinde MAPE : {mape_ratio:.3f}%")
print(f"Min H (test): {H_test.min():.1f} W/m²")

# Ay bazlı MAPE
monthly_mape = pd.DataFrame({
    'month': test_df['month'].values,
    'H':     H_test,
    'MAPE':  np.abs(H_test - pred_store[('Proposed_sqrt','PSO')]['H_te']) / H_test * 100
})
print("\nAy bazlı MAPE (H ölçeği):")
print(monthly_mape[['month','H','MAPE']].to_string(index=False))

# ── 15. TFT ──────────────────────────────────────────────────
print("\n── TFT EĞİTİMİ ──")
tft_meta    = {"status":"skipped","reason":"",
               "test_R2":np.nan,"test_RMSE":np.nan,"test_MAE":np.nan}
tft_pred_df = None
tft_seed_rows = []

if RUN_TFT and HAS_DARTS:
    try:
        df_tft = monthly_df[["date","H"]].copy()
        df_tft["Date"] = df_tft["date"]
        series = TimeSeries.from_dataframe(df_tft, "Date", "H",
                                           fill_missing_dates=False, freq="MS")
        scaler        = Scaler()
        series_scaled = scaler.fit_transform(series)
        add_month     = datetime_attribute_timeseries(
            series, attribute="month", one_hot=True)
        scaler_covs  = Scaler()
        covs_scaled  = scaler_covs.fit_transform(add_month)
        train_ts     = series_scaled[:-TEST_SIZE]

        for tft_seed in range(N_SEEDS_TFT):
            tft_model = TFTModel(
                input_chunk_length  = TFT_CFG["input_chunk_length"],
                output_chunk_length = TFT_CFG["output_chunk_length"],
                hidden_size         = TFT_CFG["hidden_size"],
                lstm_layers         = TFT_CFG["lstm_layers"],
                num_attention_heads = TFT_CFG["num_attention_heads"],
                dropout             = TFT_CFG["dropout"],
                batch_size          = TFT_CFG["batch_size"],
                n_epochs            = TFT_CFG["n_epochs"],
                add_relative_index  = True,
                optimizer_kwargs    = {"lr": 1e-3},
                random_state        = tft_seed,
                force_reset         = True,
                pl_trainer_kwargs   = {
                    "enable_checkpointing": False,
                    "logger": False,
                    "enable_model_summary": False,
                },
            )
            tft_model.fit(train_ts, future_covariates=covs_scaled, verbose=False)
            pred_scaled = tft_model.predict(
                TEST_SIZE, series=train_ts, future_covariates=covs_scaled)
            pred_H    = scaler.inverse_transform(pred_scaled)
            tft_preds = pred_H.values().flatten()
            mt = metrics(H_test, tft_preds)
            tft_seed_rows.append(
                {"seed": tft_seed, **{f"test_{k}": v for k, v in mt.items()}})
            print(f"  TFT seed={tft_seed}  R²={mt['R2']:.4f}  "
                  f"RMSE={mt['RMSE']:.2f}  MAPE={mt['MAPE']:.2f}%")

            if tft_seed == 0:
                tft_meta.update({
                    "status": "ok",
                    "test_R2":  mt["R2"],  "test_RMSE": mt["RMSE"],
                    "test_MAE": mt["MAE"], "test_MAPE": mt["MAPE"],
                    "test_MBE": mt["MBE"],
                    **TFT_CFG,
                    "future_covariates": "one_hot_month_12dim"
                })
                tft_pred_df = pd.DataFrame({
                    "date":       test_df["date"].values,
                    "H_obs":      H_test,
                    "H_pred_TFT": tft_preds,
                })
                # ── /H0 ölçeğine dönüştür, MBE hesapla ──
                tft_pred_df["HH0_obs"]  = H_test / H0_test
                tft_pred_df["HH0_pred"] = tft_pred_df["H_pred_TFT"] / H0_test

                tft_mbe_ratio  = float(np.mean(tft_pred_df["HH0_pred"] - tft_pred_df["HH0_obs"]))
                tft_rmse_ratio = float(np.sqrt(np.mean(
                    (tft_pred_df["HH0_pred"] - tft_pred_df["HH0_obs"])**2)))
                tft_mape_ratio = float(np.mean(np.abs(
                    (tft_pred_df["HH0_obs"] - tft_pred_df["HH0_pred"])
                    / tft_pred_df["HH0_obs"])) * 100)

                print(f"TFT (H/H0 ölçeği) — MBE : {tft_mbe_ratio:.5f}")
                print(f"TFT (H/H0 ölçeği) — RMSE: {tft_rmse_ratio:.5f}")
                print(f"TFT (H/H0 ölçeği) — MAPE: {tft_mape_ratio:.2f}%")
    except Exception as e:
        tft_meta["status"] = "failed"; tft_meta["reason"] = repr(e)
        print(f"  TFT hata: {repr(e)[:80]}")
else:
    tft_meta["reason"] = TFT_IMPORT_ERROR or "darts/torch mevcut değil"
    print(f"  TFT atlandı: {tft_meta['reason'][:60]}")

tft_seed_df = pd.DataFrame(tft_seed_rows)

# ── 11b. TFT WALK-FORWARD CV (KISMİ — min 36 ay kısıtı) ──────
if HAS_DARTS:
    print("\n── TFT WALK-FORWARD CV (kısmi, fold_end >= 36) ──")
    tft_wf_rows = []
    tft_wf_min  = TFT_CFG["input_chunk_length"] + 12  # 36

    for fold_end in range(tft_wf_min + 1, total_n - TEST_SIZE):
        wf_train_ts_full = series_scaled[:fold_end]
        covs_full        = covs_scaled  # tüm seri
        wf_test_idx      = fold_end

        try:
            m_wf = TFTModel(
                input_chunk_length  = TFT_CFG["input_chunk_length"],
                output_chunk_length = TFT_CFG["output_chunk_length"],
                hidden_size         = TFT_CFG["hidden_size"],
                lstm_layers         = TFT_CFG["lstm_layers"],
                num_attention_heads = TFT_CFG["num_attention_heads"],
                dropout             = TFT_CFG["dropout"],
                batch_size          = TFT_CFG["batch_size"],
                n_epochs            = TFT_CFG["n_epochs"],
                add_relative_index  = True,
                random_state        = 42,
                force_reset         = True,
                pl_trainer_kwargs   = {"enable_checkpointing": False,
                                       "logger": False,
                                       "enable_model_summary": False},
            )
            m_wf.fit(wf_train_ts_full, future_covariates=covs_full, verbose=False)
            pred_wf  = m_wf.predict(1, series=wf_train_ts_full,
                                    future_covariates=covs_full)
            H_pred_wf = float(scaler.inverse_transform(pred_wf).values().flatten()[0])
            H_true_wf = monthly_df.iloc[wf_test_idx]["H"]
            tft_wf_rows.append({
                "fold": fold_end,
                "H_true": H_true_wf, "H_pred": H_pred_wf,
                "abs_error": abs(H_pred_wf - H_true_wf),
                "error": H_pred_wf - H_true_wf
            })
            print(f"  TFT WF fold={fold_end}  pred={H_pred_wf:.1f}  true={H_true_wf:.1f}")
        except Exception as e_wf:
            print(f"  TFT WF fold={fold_end} HATA: {repr(e_wf)[:60]}")

    tft_wf_df = pd.DataFrame(tft_wf_rows)
    if len(tft_wf_df):
        tft_wf_rmse = float(np.sqrt(np.mean(tft_wf_df["error"]**2)))
        tft_wf_mae  = float(np.mean(tft_wf_df["abs_error"]))
        tft_wf_mbe  = float(np.mean(tft_wf_df["error"]))
        tft_wf_mape = float(np.mean(tft_wf_df["abs_error"] /
                                    np.abs(tft_wf_df["H_true"])) * 100)
        print(f"\nTFT Walk-Forward (n={len(tft_wf_df)} katlama): "
              f"RMSE={tft_wf_rmse:.2f}  MAE={tft_wf_mae:.2f}  "
              f"MBE={tft_wf_mbe:.2f}  MAPE={tft_wf_mape:.2f}%")
        save_csv(tft_wf_df, f"{tables_dir}/13_tft_walkforward.csv")

# ── TFT HİPERPARAMETRE IZGARA ARAMASI ──────────────
if HAS_DARTS:
    from itertools import product as iproduct
    print("\n── TFT GRID SEARCH ──")
    hidden_sizes_gs = [4, 8, 16, 32]          # +32 eklendi
    dropouts_gs     = [0.05, 0.1, 0.2]
    n_epochs_gs     = [150, 300]               # +epoch ekseni
    test_seeds_gs   = [0, 7, 11, 15, 19]

    for hs, dr in iproduct(hidden_sizes_gs, dropouts_gs):
        scores_gs = []
        for seed_gs in test_seeds_gs:
            try:
                m_gs = TFTModel(
                    input_chunk_length  = TFT_CFG['input_chunk_length'],
                    output_chunk_length = TFT_CFG['output_chunk_length'],
                    hidden_size         = hs,
                    lstm_layers         = TFT_CFG['lstm_layers'],
                    num_attention_heads = TFT_CFG['num_attention_heads'],
                    dropout             = dr,
                    batch_size          = TFT_CFG['batch_size'],
                    n_epochs            = ep,
                    random_state        = seed_gs,
                )
                m_gs.fit(train_ts,
                        future_covariates=covs_scaled,
                        verbose=False)
                p_gs    = m_gs.predict(TEST_SIZE,
                                      series=train_ts,
                                      future_covariates=covs_scaled)
                pred_gs = scaler.inverse_transform(p_gs).values().flatten()
                scores_gs.append(float(r2_score(H_test, pred_gs)))
            except Exception as e_gs:
                scores_gs.append(np.nan)
        grid_results.append({
            'hidden_size': hs, 'dropout': dr,
            'mean_R2':     np.nanmean(scores_gs),
            'std_R2':      np.nanstd(scores_gs),
        })
        print(f"  hidden={hs:2d}  dropout={dr:.2f}  "
              f"mean_R2={np.nanmean(scores_gs):.4f}  "
              f"std_R2={np.nanstd(scores_gs):.4f}")

    df_grid_tft = (pd.DataFrame(grid_results)
                   .sort_values('mean_R2', ascending=False)
                   .reset_index(drop=True))
    save_csv(df_grid_tft,
             f"{OUTPUT_ROOT}/tables/tft_grid_search.csv")
    print("\nEn iyi TFT konfigürasyonu:")
    print(df_grid_tft.head(3).round(4).to_string(index=False))

# ── 16. NİHAİ TABLO ──────────────────────────────────────────
tft_row = pd.DataFrame([{
    "Model": "TFT", "Method": "DeepLearning", "Origin": "notebook",
    "n_params": 5900,
    "train_R2": np.nan, "train_RMSE": np.nan,
    "train_MAE": np.nan, "train_MAPE": np.nan, "train_MBE": np.nan,
    "test_R2":   tft_meta["test_R2"],
    "test_RMSE": tft_meta["test_RMSE"],
    "test_MAE":  tft_meta["test_MAE"],
    "test_MAPE": tft_meta.get("test_MAPE", np.nan),
    "test_MBE":  tft_meta.get("test_MBE",  np.nan),
    "obj_train_SSE_ratio": np.nan,
    "params": str({k: TFT_CFG[k] for k in TFT_CFG}),
}])

final_df = (pd.concat([results_df, tft_row], ignore_index=True)
            .sort_values(["test_R2","test_RMSE"], ascending=[False,True])
            .reset_index(drop=True))

# ── 17. GRAFİKLER ────────────────────────────────────────────
plots_dir  = safe_mkdir(f"{OUTPUT_ROOT}/plots")
tables_dir = safe_mkdir(f"{OUTPUT_ROOT}/tables")

# G1: Test R² barplot
# G1: Test R² barplot — düzeltilmiş eksen
tmp = final_df.dropna(subset=["test_R2"]).copy()
tmp = tmp.sort_values("test_R2", ascending=True).reset_index(drop=True)
tmp["label"] = tmp["Model"] + " | " + tmp["Method"]

palette = {"paper": "#4C72B0", "proposed": "#DD8452", "notebook": "#55A868"}
colors  = [palette.get(o, "#8C8C8C") for o in tmp["Origin"]]

fig, ax = plt.subplots(figsize=(14, max(6, len(tmp)*0.35)))
bars = ax.barh(tmp["label"], tmp["test_R2"], color=colors)

# Eksen sınırlarını R² değer aralığına göre ayarla
r2_min = max(0.0, tmp["test_R2"].min() - 0.02)
r2_max = min(1.0, tmp["test_R2"].max() + 0.01)
ax.set_xlim(r2_min, r2_max)

# Her çubuğun yanına değer yaz
for bar, val in zip(bars, tmp["test_R2"]):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2,
            f"{val:.4f}", va="center", ha="left", fontsize=8)

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=v, label=k) for k, v in palette.items()]
ax.legend(handles=legend_elements, loc="lower right")

ax.set_xlabel("Test $R^2$", fontsize=11)
ax.set_ylabel("")
ax.set_title("Test $R^2$ — tüm model/yöntem kombinasyonları", fontsize=12)
ax.xaxis.set_major_formatter(
    plt.FuncFormatter(lambda x, _: f"{x:.3f}"))
plt.tight_layout()
plt.savefig(f"{plots_dir}/01_test_r2_all_models.png",
            dpi=150, bbox_inches="tight")
plt.close()

# G2: Top-5 observed vs predicted
top5 = final_df.dropna(subset=["test_R2"]).head(5)
plt.figure(figsize=(14, 6))
plt.plot(test_df["date"], H_test, marker="o", lw=2.5, label="Observed", color="black")
for _, r in top5.iterrows():
    key = (r["Model"], r["Method"])
    if key in pred_store:
        plt.plot(test_df["date"], pred_store[key]["H_te"],
                 marker="o", lw=1.5, label=f"{r['Model']} {r['Method']}")
    elif r["Model"] == "TFT" and tft_pred_df is not None:
        plt.plot(tft_pred_df["date"], tft_pred_df["H_pred_TFT"],
                 marker="s", lw=1.8, linestyle="--", label="TFT")
plt.title("Test dönemi: gözlenen vs en iyi 5 model")
plt.xticks(rotation=45); plt.ylabel("H (W/m²)")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(f"{plots_dir}/02_test_observed_vs_top5.png"); plt.close()

# G3: Multi-seed boxplot
melt = multi_df.melt(
    id_vars=["method","seed"],
    value_vars=["test_R2","test_RMSE","test_MAE","test_MAPE"],
    var_name="metric", value_name="value")
g = sns.catplot(data=melt, x="method", y="value", col="metric",
                kind="box", sharey=False, height=5, aspect=0.9)
g.fig.suptitle("Proposed_sqrt — 20-seed dağılımları", y=1.04)
plt.tight_layout()
plt.savefig(f"{plots_dir}/03_multiseed_boxplots.png"); plt.close()

# G4: Seed bazlı R²
plt.figure(figsize=(12, 5))
for m in ["PSO","POA"]:
    sub = multi_df[multi_df["method"] == m].sort_values("seed")
    plt.plot(sub["seed"], sub["test_R2"], marker="o", label=m)
plt.title("Seed bazında test R² (Proposed_sqrt)")
plt.xlabel("Seed"); plt.ylabel("Test R²"); plt.legend()
plt.tight_layout()
plt.savefig(f"{plots_dir}/04_seedwise_test_r2.png"); plt.close()

# G5: Yakınsama eğrisi
if len(hist_df):
    mean_hist = hist_df.groupby(["method","iter"], as_index=False)["best_sse"].mean()
    plt.figure(figsize=(12, 5))
    for m in ["PSO","POA"]:
        sub = mean_hist[mean_hist["method"] == m]
        plt.plot(sub["iter"], sub["best_sse"], label=m)
    plt.title("Ortalama yakınsama eğrisi (PSO vs POA)")
    plt.xlabel("İterasyon"); plt.ylabel("Mean best SSE"); plt.legend()
    plt.tight_layout()
    plt.savefig(f"{plots_dir}/05_mean_convergence.png"); plt.close()

# G6: TFT tahmin grafiği
if tft_pred_df is not None:
    plt.figure(figsize=(12, 5))
    plt.plot(tft_pred_df["date"], tft_pred_df["H_obs"],
             marker="o", lw=2.5, label="Observed", color="blue")
    plt.plot(tft_pred_df["date"], tft_pred_df["H_pred_TFT"],
             marker="s", lw=1.8, linestyle="--", label="TFT", color="red")
    r2s = (f"R²={tft_meta['test_R2']:.4f}"
           if not np.isnan(tft_meta["test_R2"]) else "")
    plt.title(f"TFT Model — Son {TEST_SIZE} ay tahmin  {r2s}")
    plt.xlabel("Tarih"); plt.ylabel("H (W/m²)")
    plt.xticks(rotation=45); plt.legend(); plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{plots_dir}/06_tft_test_forecast.png", dpi=150); plt.close()

# G7: Residual plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (mname, method, color) in zip(axes, [
        ("Proposed_sqrt","PSO","steelblue"),
        ("Proposed_sqrt","POA","darkorange")]):
    key = (mname, method)
    if key in pred_store:
        resid = pred_store[key]["H_te"] - H_test
        ax.scatter(pred_store[key]["H_te"], resid, color=color, alpha=0.8)
        ax.axhline(0, color="black", lw=1.5, linestyle="--")
        ax.set_title(f"Residuals — {mname} {method}")
        ax.set_xlabel("Predicted H (W/m²)")
        ax.set_ylabel("Residual (Pred − Obs)")
plt.tight_layout()
plt.savefig(f"{plots_dir}/07_residual_plots.png"); plt.close()

# G7b: Scatter plot — Proposed_sqrt PSO ve POA
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (mname, method, color) in zip(axes, [
        ("Proposed_sqrt", "PSO", "steelblue"),
        ("Proposed_sqrt", "POA", "darkorange")]):
    key = (mname, method)
    if key in pred_store:
        # Eğitim + test birleşik
        H_all_obs  = np.concatenate([H_train, H_test])
        H_all_pred = np.concatenate([
            pred_store[key]["H_tr"],
            pred_store[key]["H_te"]
        ])
        ax.scatter(H_all_obs[:len(H_train)], H_all_pred[:len(H_train)],
                   color=color, alpha=0.5, label="Eğitim", marker="o")
        ax.scatter(H_all_obs[len(H_train):], H_all_pred[len(H_train):],
                   color="red", alpha=0.9, label="Test", marker="^", s=80)
        # 1:1 referans çizgisi
        lims = [min(H_all_obs.min(), H_all_pred.min()) - 50,
                max(H_all_obs.max(), H_all_pred.max()) + 50]
        ax.plot(lims, lims, "k--", lw=1.5, label="1:1")
        ax.set_xlim(lims); ax.set_ylim(lims)
        ax.set_xlabel("Gözlenen H (W/m²)")
        ax.set_ylabel("Tahmin Edilen H (W/m²)")
        ax.set_title(f"Proposed_sqrt — {method}")
        ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(f"{plots_dir}/07b_scatter_obs_pred.png",
            dpi=150, bbox_inches="tight")
plt.close()

# G8: Walk-forward RMSE
if len(wf_summary):
    plt.figure(figsize=(10, 5))
    wf_pivot = wf_summary.pivot(index="model", columns="method", values="WF_RMSE")
    wf_pivot.plot(kind="bar", ax=plt.gca())
    plt.title("Walk-Forward RMSE Karşılaştırması")
    plt.xlabel("Model"); plt.ylabel("WF-RMSE (W/m²)")
    plt.xticks(rotation=30); plt.tight_layout()
    plt.savefig(f"{plots_dir}/08_walkforward_rmse.png"); plt.close()

# G9: TFT multi-seed
if len(tft_seed_df):
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    for ax, metric in zip(axes, ["test_R2","test_RMSE","test_MAE"]):
        ax.bar(tft_seed_df["seed"], tft_seed_df[metric], color="salmon")
        ax.axhline(tft_seed_df[metric].mean(),
                   color="red", linestyle="--", label="Mean")
        ax.set_title(f"TFT — {metric}")
        ax.set_xlabel("Seed"); ax.legend()
    plt.suptitle(f"TFT {N_SEEDS_TFT}-Seed Kararlılık Analizi", y=1.02)
    plt.tight_layout()
    plt.savefig(f"{plots_dir}/09_tft_multiseed.png"); plt.close()

# G10: Bootstrap katsayı dağılımları (4 parametre)
if len(bs_df):
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    for ax, coef in zip(axes, COEF_NAMES):
        col = f"coef_{coef}"
        ax.hist(bs_df[col], bins=30, edgecolor="white",
                color="steelblue", alpha=0.8)
        row_ci = ci_df[ci_df["coef"] == coef].iloc[0]
        ax.axvline(row_ci["mean"],    color="red",    lw=2, label="Mean")
        ax.axvline(row_ci["ci_low"],  color="orange", lw=1.5,
                   linestyle="--", label="95% CI")
        ax.axvline(row_ci["ci_high"], color="orange", lw=1.5, linestyle="--")
        ax.set_title(f"Bootstrap — coef {coef}")
        ax.legend(fontsize=7)
    plt.suptitle(
        "Proposed_sqrt PSO — Bootstrap Katsayı Dağılımları\n"
        "H/H0 = a + b·x + c·x² + d·√x  (N=500)",
        y=1.04)
    plt.tight_layout()
    plt.savefig(f"{plots_dir}/10_bootstrap_coeff_dist.png"); plt.close()

# ── G11: S/S0 REJİMİNE GÖRE ARTIK ANALİZİ ──
models_compare = [
    ('Proposed_sqrt', 'PSO',   'M7 (PSO)',   'steelblue'),
    ('Linear',        'EKK',   'M1 (EKK)',   'gray'),
    ('Polynomial3',   'EKK',   'M3 (EKK)',   'darkorange'),
]

df_resid = test_df[['month','SoverS0','H','H0']].copy()
for mname, method, label, _ in models_compare:
    key = (mname, method)
    if key in pred_store:
        df_resid[f'pred_{label}'] = pred_store[key]['H_te']
        df_resid[f'abs_err_{label}'] = np.abs(pred_store[key]['H_te'] - H_test)

bins   = [0.0, 0.35, 0.60, 1.0]
labels_bin = ['Düşük\n(S/S₀<0.35)', 'Orta\n(0.35–0.60)', 'Yüksek\n(>0.60)']
df_resid['SS0_group'] = pd.cut(df_resid['SoverS0'], bins=bins, labels=labels_bin)

fig, ax = plt.subplots(figsize=(9, 5))
x_pos = np.arange(len(labels_bin))
width = 0.25

for i, (mname, method, label, color) in enumerate(models_compare):
    grp_means = df_resid.groupby('SS0_group', observed=True)[f'abs_err_{label}'].mean()
    ax.bar(x_pos + i*width, grp_means.values, width,
           label=label, color=color, alpha=0.85, edgecolor='white')

ax.set_xticks(x_pos + width)
ax.set_xticklabels(labels_bin, fontsize=10)
ax.set_xlabel('S/S₀ Rejimi', fontsize=11)
ax.set_ylabel('Ortalama Mutlak Hata (W/m²)', fontsize=11)
ax.set_title('S/S₀ Rejimine Göre Model Artıkları (Test Kümesi)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(f'{plots_dir}/11_residual_by_ss0_group.png', dpi=150)
plt.show()
print("✓ 11_residual_by_ss0_group.png kaydedildi")

# ── 18. CSV + PNG KAYDET ─────────────────────────────────────
save_csv(monthly_df,               f"{tables_dir}/01_monthly_data.csv")
save_csv(results_df,               f"{tables_dir}/02_empirik_results.csv")
save_csv(final_df,                 f"{tables_dir}/03_FINAL_all_models_with_TFT.csv")
save_csv(summary_df,               f"{tables_dir}/04_multiseed_summary.csv")
save_csv(multi_df,                 f"{tables_dir}/05_multiseed_runs.csv")
save_csv(stats_df,                 f"{tables_dir}/07_statistical_tests_cohens_d.csv")
save_csv(pd.DataFrame([tft_meta]), f"{tables_dir}/08_tft_config_metrics.csv")
save_csv(wf_summary,               f"{tables_dir}/09_walkforward_summary.csv")
save_csv(ci_df,                    f"{tables_dir}/10_bootstrap_ci.csv")
save_csv(tft_seed_df,              f"{tables_dir}/11_tft_multiseed.csv")
if tft_pred_df is not None:
    save_csv(tft_pred_df,          f"{tables_dir}/12_tft_predictions.csv")

save_df_as_png(final_df.round(4),
               f"{tables_dir}/03_FINAL_all_models_with_TFT.png",
               "Nihai Karşılaştırma Tablosu (MAPE+MBE dahil)", max_rows=50)
save_df_as_png(summary_df.round(4),
               f"{tables_dir}/04_multiseed_summary.png",
               "Proposed_sqrt 20-Seed Özeti")
save_df_as_png(stats_df.round(4),
               f"{tables_dir}/07_statistical_tests.png",
               "PSO vs POA İstatistiksel Testler (Cohen's d dahil)")
save_df_as_png(ci_df.round(4),
               f"{tables_dir}/10_bootstrap_ci.png",
               "Bootstrap 95% CI — Proposed_sqrt PSO\nH/H0=a+b·x+c·x²+d·√x")
save_df_as_png(wf_summary.round(2),
               f"{tables_dir}/09_walkforward_summary.png",
               "Walk-Forward CV Sonuçları")

# ── 19. DRIVE KOPYASI ─────────────────────────────────────────
if SAVE_TO_DRIVE_TOO and IN_COLAB and os.path.exists("/content/drive/MyDrive"):
    import shutil
    if os.path.exists(DRIVE_OUTPUT_ROOT):
        shutil.rmtree(DRIVE_OUTPUT_ROOT)
    shutil.copytree(OUTPUT_ROOT, DRIVE_OUTPUT_ROOT)
    print(f"\n✓ Çıktılar Drive'a kopyalandı: {DRIVE_OUTPUT_ROOT}")

# ── 20. SON ÖZET ──────────────────────────────────────────────
print("\n" + "="*65)
print("ÖZET SONUÇLAR")
print("="*65)
print(final_df[["Model","Method","test_R2","test_RMSE",
                "test_MAE","test_MAPE","test_MBE"]]
      .head(10).to_string(index=False))
print("="*65)
print("\nWalk-Forward CV Özeti:")
print(wf_summary.round(2).to_string(index=False))
print("\nBootstrap 95% CI — H/H0 = a + b·x + c·x² + d·√x:")
print(ci_df.round(4).to_string(index=False))
print("\nCohen's d (PSO vs POA):")
print(stats_df[["metric","cohens_d","ttest_p","wilcoxon_p"]]
      .round(4).to_string(index=False))
print(f"\n✓ Çıktı klasörü : {OUTPUT_ROOT}")
print(f"  plots/  → 10 grafik PNG")
print(f"  tables/ → 12 tablo CSV + 5 PNG")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Excel bulundu: /content/drive/MyDrive/solar_gunler_Serkan.xlsx
✓ Sütun eşlemesi:
   year   → Yıl
   month  → Ay
   day    → Gün
   H      → Toplam Küresel Güneş Radyasyonu (Watt/m²)
   S      → Güneşlenme Süresi (Saat) S
   S0     → Gün
 Uzunlugu (saat) So
   H0     → Ho w/m2
   S/S0   → S/So
   H/H0   → H/Ho

✓ Veri hazır: 75 aylık gözlem  [2018-01-01 → 2024-03-01]
   Eğitim: 63  |  Test: 12

── EMPİRİK MODEL EĞİTİMİ ──
  Linear           EKK
  Linear           PSO
  Linear           POA
  Polynomial2      EKK
  Polynomial2      PSO
  Polynomial2      POA
  Polynomial3      EKK
  Polynomial3      PSO
  Polynomial3      POA
  Exponential      EKK
  Exponential      PSO
  Exponential      POA
  Logarithmic      EKK
  Logarithmic      PSO
  Logarithmic      POA
  Power            EKK
  Power            PSO
  Power            POA
  Proposed_sqrt    EKK
  Propo

INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=300` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs


Predicting: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs


  TFT seed=0  R²=0.9236  RMSE=300.71  MAPE=7.91%
TFT (H/H0 ölçeği) — MBE : -0.00861
TFT (H/H0 ölçeği) — RMSE: 0.03250
TFT (H/H0 ölçeği) — MAPE: 7.91%


INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=300` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs


Predicting: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs


  TFT seed=1  R²=0.9522  RMSE=237.89  MAPE=4.73%
